# Load Dataset  
Load dataset and create pandas dataframe.

In [1]:
import pandas as pd
import zipfile

zip = zipfile.ZipFile(r'dataset.zip')
zip.extractall(r'./')
df = pd.read_csv("dataset.csv")
print(df[:3])

        SrcAddr        DstAddr Proto  Sport  Dport State  sTos  dTos  \
0  31.96.153.11  147.32.84.229   tcp  60257    443   RST   0.0   0.0   
1  83.228.37.92  147.32.84.229   tcp   2571  13363   RST   0.0   0.0   
2  83.228.37.92  147.32.84.229   tcp   2574    443   RST   0.0   0.0   

      SrcWin   DstWin  ...  SAppBytes  DAppBytes       Dur TotPkts  TotBytes  \
0  2097152.0  65535.0  ...          0          0  9.016532       7       508   
1    65535.0  65535.0  ...          0          0  2.903761       3       184   
2    65535.0  65535.0  ...          0          0  3.032142       3       184   

   TotAppByte      Rate   SrcRate   DstRate                            Label  
0           0  0.665444  0.221815  0.336762  flow=Background-TCP-Established  
1           0  0.688762  0.344381  0.000000  flow=Background-TCP-Established  
2           0  0.659600  0.329800  0.000000  flow=Background-TCP-Established  

[3 rows x 33 columns]


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── 1. Fix label format (strip "flow=" prefix) ────────────────────────────────
df['Label'] = df['Label'].str.replace('flow=', '', regex=False)

print("Botnet flows:", df['Label'].str.contains('From-Botnet').sum())
print("Normal flows:", df['Label'].str.contains('From-Normal').sum())
print("Background:  ", df['Label'].str.contains('Background').sum())

# ── 2. Filter & label ─────────────────────────────────────────────────────────
# Keep From-Botnet (malicious) and From-Normal (benign)
# Drop Background and To-* (noisy / ambiguous)
df = df[df['Label'].str.contains('From-Botnet|From-Normal')]
df['target'] = df['Label'].str.contains('From-Botnet').astype(int)
print("\nClass balance after filter:")
print(df['target'].value_counts())

# ── 3. Drop identifier & leaky columns ───────────────────────────────────────
drop_cols = ['SrcAddr', 'DstAddr', 'StartTime', 'LastTime', 'Label', 'State']
df.drop(columns=drop_cols, inplace=True)

# ── 4. Fix hex ports ──────────────────────────────────────────────────────────
def parse_port(x):
    try:
        if isinstance(x, str) and x.startswith('0x'):
            return int(x, 16)
        return float(x)
    except:
        return -1

df['Sport'] = df['Sport'].apply(parse_port)
df['Dport'] = df['Dport'].apply(parse_port)

# ── 5. Encode protocol ────────────────────────────────────────────────────────
df['Proto'] = df['Proto'].str.lower().str.strip()
df = pd.get_dummies(df, columns=['Proto'], drop_first=True)

# ── 6. Fill missing values ────────────────────────────────────────────────────
# sTos/dTos, SrcWin/DstWin etc. can be NaN for non-TCP flows
df['sTos']   = df['sTos'].fillna(0)
df['dTos']   = df['dTos'].fillna(0)
df['SrcWin'] = df['SrcWin'].fillna(0)
df['DstWin'] = df['DstWin'].fillna(0)
df['sHops']  = df['sHops'].fillna(0)
df['dHops']  = df['dHops'].fillna(0)
df['TcpRtt'] = df['TcpRtt'].fillna(0)
df['SynAck'] = df['SynAck'].fillna(0)
df['AckDat'] = df['AckDat'].fillna(0)
df['sTtl']   = df['sTtl'].fillna(0)
df['dTtl']   = df['dTtl'].fillna(0)

# ── 7. Derive behavioural features ───────────────────────────────────────────
df['BytesPerPkt']    = df['TotBytes']   / (df['TotPkts']  + 1e-9)
df['BytesPerSec']    = df['TotBytes']   / (df['Dur']      + 1e-9)
df['PktsPerSec']     = df['TotPkts']    / (df['Dur']      + 1e-9)
df['SrcByteRatio']   = df['SrcBytes']   / (df['TotBytes'] + 1e-9)
df['AppByteRatio']   = df['TotAppByte'] / (df['TotBytes'] + 1e-9)  # payload ratio
df['PktAsymmetry']   = df['SrcPkts']    / (df['TotPkts']  + 1e-9)  # src/total pkts
df['is_well_known_dst'] = (df['Dport'] < 1024).astype(int)
df['is_high_dst_port']  = (df['Dport'] > 49151).astype(int)

# ── 8. Drop any leftover non-numeric columns ──────────────────────────────────
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print("\nDropping remaining non-numeric:", non_numeric)
    df.drop(columns=non_numeric, inplace=True)

df.dropna(inplace=True)
print("\nFinal shape:", df.shape)
print("Features:", df.drop('target', axis=1).columns.tolist())

# ── 9. Split (no shuffle — preserve time order) ───────────────────────────────
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# ── 10. Scale ─────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("\nX_train:", X_train.shape, "| X_test:", X_test.shape)
print("Ready to train!")

Botnet flows: 20941
Normal flows: 9082
Background:   1778061

Class balance after filter:
target
1    20941
0     9082
Name: count, dtype: int64

Dropping remaining non-numeric: ['Proto_icmp', 'Proto_tcp', 'Proto_udp']

Final shape: (30014, 35)
Features: ['Sport', 'Dport', 'sTos', 'dTos', 'SrcWin', 'DstWin', 'sHops', 'dHops', 'sTtl', 'dTtl', 'TcpRtt', 'SynAck', 'AckDat', 'SrcPkts', 'DstPkts', 'SrcBytes', 'DstBytes', 'SAppBytes', 'DAppBytes', 'Dur', 'TotPkts', 'TotBytes', 'TotAppByte', 'Rate', 'SrcRate', 'DstRate', 'BytesPerPkt', 'BytesPerSec', 'PktsPerSec', 'SrcByteRatio', 'AppByteRatio', 'PktAsymmetry', 'is_well_known_dst', 'is_high_dst_port']

X_train: (24011, 34) | X_test: (6003, 34)
Ready to train!
